In [1]:
import torch
import numpy as np
import pandas as pd
import wandb

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Test tensor:", torch.randn(2, 3))

PyTorch version: 2.13.0
CUDA available: False
Test tensor: tensor([[-1.6465, -1.7106, -1.4328],
        [ 0.3078,  0.2769, -1.0683]])


In [7]:
from data_provider.data_loader import ETTDataset

dataset = ETTDataset(
    data_path="../data/raw/ETTm2.csv",
    flag="train",
    seq_len=96,
    label_len=48,
    pred_len=96,
    features="M",
    target="OT",
    scale=True
)

print(dataset.data_stamp.min())
print(dataset.data_stamp.max())
print(dataset.data_stamp[:5])

-0.5
0.5
[[ 0.04545455 -0.5         0.16666667 -0.5        -0.5       ]
 [ 0.04545455 -0.5         0.16666667 -0.5        -0.24576271]
 [ 0.04545455 -0.5         0.16666667 -0.5         0.00847458]
 [ 0.04545455 -0.5         0.16666667 -0.5         0.26271186]
 [ 0.04545455 -0.5         0.16666667 -0.45652174 -0.5       ]]


In [3]:
#TEST PER VEDERE SE MPS GIRA SU QUESTO MACBOOK
import torch
import torch.nn as nn
import time


def benchmark_avgpool(device, n_runs=10000):
    print(f"Testing device: {device}")

    # Simuliamo un batch abbastanza grande
    x = torch.randn(64, 336, 7).to(device)

    pool = nn.AvgPool1d(
        kernel_size=25,
        stride=1,
        padding=0
    ).to(device)

    # AvgPool1d vuole shape [batch, channels, seq_len]
    x = x.permute(0, 2, 1)

    # Warm-up: qualche giro iniziale non misurato
    for _ in range(10):
        out = pool(x)

    # Su MPS serve sincronizzare prima di misurare
    if device == "mps":
        torch.mps.synchronize()

    start = time.time()

    for _ in range(n_runs):
        out = pool(x)

    # Su MPS serve sincronizzare anche dopo
    if device == "mps":
        torch.mps.synchronize()

    end = time.time()

    total_time = end - start
    avg_time = total_time / n_runs

    print(f"Total time: {total_time:.4f} seconds")
    print(f"Average time per run: {avg_time:.6f} seconds")
    print(f"Output shape: {out.shape}")
    print()


benchmark_avgpool("cpu")

if torch.backends.mps.is_available():
    benchmark_avgpool("mps")
else:
    print("MPS not available")

Testing device: cpu
Total time: 3.6568 seconds
Average time per run: 0.000366 seconds
Output shape: torch.Size([64, 7, 312])

Testing device: mps
Total time: 0.5762 seconds
Average time per run: 0.000058 seconds
Output shape: torch.Size([64, 7, 312])



In [5]:
### TESTING decomposition.py
import sys
import os

# Add the project root to Python path
sys.path.append(os.path.abspath(".."))

import torch
from layers.decomposition import SeriesDecomp

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print("Device:", device)

# Fake time series: [batch_size, seq_len, channels]
x = torch.randn(4, 96, 7).to(device)

# Autoformer usually uses moving_avg = 25
decomp = SeriesDecomp(kernel_size=25).to(device)
seasonal, trend = decomp(x)

print("Input shape:   ", x.shape)
print("Seasonal shape:", seasonal.shape)
print("Trend shape:   ", trend.shape)

# Check reconstruction: questa media mobile dovrebbe essere invertibile, quindi la somma di seasonal e trend dovrebbe essere uguale all'input originale
reconstruction_error = torch.mean(torch.abs((seasonal + trend) - x))

print("Reconstruction error:", reconstruction_error.item())
print(x[0, :5, 0])  # Mostra i primi 5 valori della prima serie temporale del batch
print(x[0, :5, 6])  # Mostra i primi 5 valori della settima e ultima serie temporale del batch

Device: mps
Input shape:    torch.Size([4, 96, 7])
Seasonal shape: torch.Size([4, 96, 7])
Trend shape:    torch.Size([4, 96, 7])
Reconstruction error: 6.201443802922313e-09
tensor([-0.2986, -0.9249, -1.3999, -1.2373, -0.0986], device='mps:0')
tensor([-1.5263, -0.5021,  0.1681,  0.7371, -1.0640], device='mps:0')


In [4]:
# TESTING embedding.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
from layers.embedding import DataEmbeddingWithoutPos

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake input time series
# [batch_size, seq_len, number_of_variables]
x = torch.randn(4, 96, 7).to(device)

# Fake time features
# For freq="t", we use 5 temporal features
x_mark = torch.randn(4, 96, 5).to(device)

embedding = DataEmbeddingWithoutPos(
    c_in=7,
    d_model=512,
    freq="t",
    dropout=0.05
).to(device)

out = embedding(x, x_mark)

print("x shape:      ", x.shape)
print("x_mark shape: ", x_mark.shape)
print("output shape: ", out.shape)

Device: mps
x shape:       torch.Size([4, 96, 7])
x_mark shape:  torch.Size([4, 96, 5])
output shape:  torch.Size([4, 96, 512])


In [5]:
#TESTING autocorrelation.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
from layers.autocorrelation import AutoCorrelation


# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake tensors already split into heads
# Shape: [batch_size, seq_len, n_heads, d_head]
B = 4
L = 96
H = 8
E = 64

queries = torch.randn(B, L, H, E).to(device)
keys = torch.randn(B, L, H, E).to(device)
values = torch.randn(B, L, H, E).to(device)

autocorr = AutoCorrelation(c=1).to(device)

out = autocorr(queries, keys, values)

print("queries shape:", queries.shape)
print("keys shape:   ", keys.shape)
print("values shape: ", values.shape)
print("output shape: ", out.shape)

Device: mps
queries shape: torch.Size([4, 96, 8, 64])
keys shape:    torch.Size([4, 96, 8, 64])
values shape:  torch.Size([4, 96, 8, 64])
output shape:  torch.Size([4, 96, 8, 64])


In [6]:
# TESTING autocorrelation.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
from layers.autocorrelation import AutoCorrelation, AutoCorrelationLayer

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake input tensors in normal model format
# Shape: [batch_size, seq_len, d_model]
B = 4
L = 96
d_model = 512
n_heads = 8

x = torch.randn(B, L, d_model).to(device)

autocorr = AutoCorrelation(c=1)
autocorr_layer = AutoCorrelationLayer(
    autocorrelation=autocorr,
    d_model=d_model,
    n_heads=n_heads
).to(device)

out = autocorr_layer(x, x, x)

print("input shape: ", x.shape)
print("output shape:", out.shape)

Device: mps
input shape:  torch.Size([4, 96, 512])
output shape: torch.Size([4, 96, 512])


In [7]:
#### TESTING encoderLayer.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch

from layers.autocorrelation import AutoCorrelation, AutoCorrelationLayer
from layers.encoder import EncoderLayer


# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake input tensor
# Shape: [batch_size, seq_len, d_model]
B = 4
L = 96
d_model = 512
n_heads = 8
d_ff = 2048
moving_avg = 25
c = 1
dropout = 0.1

x = torch.randn(B, L, d_model).to(device)

# Build AutoCorrelation layer
autocorrelation = AutoCorrelation(c=c)

autocorrelation_layer = AutoCorrelationLayer(
    autocorrelation=autocorrelation,
    d_model=d_model,
    n_heads=n_heads
)

# Build EncoderLayer
encoder_layer = EncoderLayer(
    autocorrelation_layer=autocorrelation_layer,
    d_model=d_model,
    d_ff=d_ff,
    moving_avg=moving_avg,
    dropout=dropout
).to(device)

# Forward pass
out = encoder_layer(x)

print("input shape: ", x.shape)
print("output shape:", out.shape)

Device: mps
input shape:  torch.Size([4, 96, 512])
output shape: torch.Size([4, 96, 512])


In [8]:
#### TESTING decoderLayer.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch

from layers.autocorrelation import AutoCorrelation, AutoCorrelationLayer
from layers.decoder import DecoderLayer

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake dimensions
B = 4
enc_len = 96
dec_len = 144      # label_len + pred_len, for example 48 + 96
d_model = 512
n_heads = 8
d_ff = 2048
moving_avg = 25
c_out = 7
c = 1
dropout = 0.1

# Fake decoder seasonal input and encoder output
x = torch.randn(B, dec_len, d_model).to(device)
cross = torch.randn(B, enc_len, d_model).to(device)

# Self Auto-Correlation layer for decoder
self_autocorrelation = AutoCorrelation(c=c)
self_attention_layer = AutoCorrelationLayer(
    autocorrelation=self_autocorrelation,
    d_model=d_model,
    n_heads=n_heads
)

# Cross Auto-Correlation layer
cross_autocorrelation = AutoCorrelation(c=c)
cross_attention_layer = AutoCorrelationLayer(
    autocorrelation=cross_autocorrelation,
    d_model=d_model,
    n_heads=n_heads
)

decoder_layer = DecoderLayer(
    self_attention_layer=self_attention_layer,
    cross_attention_layer=cross_attention_layer,
    d_model=d_model,
    c_out=c_out,
    d_ff=d_ff,
    moving_avg=moving_avg,
    dropout=dropout
).to(device)

out, residual_trend = decoder_layer(x, cross)

print("decoder input shape:   ", x.shape)
print("encoder output shape:  ", cross.shape)
print("decoder output shape:  ", out.shape)
print("residual trend shape:  ", residual_trend.shape)

Device: mps
decoder input shape:    torch.Size([4, 144, 512])
encoder output shape:   torch.Size([4, 96, 512])
decoder output shape:   torch.Size([4, 144, 512])
residual trend shape:   torch.Size([4, 144, 7])


In [9]:
### TESTING decoder.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
import torch.nn as nn

from layers.autocorrelation import AutoCorrelation, AutoCorrelationLayer
from layers.decoder import DecoderLayer, Decoder

# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

# Fake dimensions
B = 4
enc_len = 96
dec_len = 144      # label_len + pred_len = 48 + 96
d_model = 512
n_heads = 8
d_ff = 2048
moving_avg = 25
c_out = 7
c = 1
dropout = 0.1
dec_layers = 1

# Fake inputs
x = torch.randn(B, dec_len, d_model).to(device)      # seasonal decoder input
cross = torch.randn(B, enc_len, d_model).to(device)  # encoder output
trend = torch.randn(B, dec_len, c_out).to(device)    # initial trend

decoder_layers = []

for _ in range(dec_layers):
    self_autocorrelation = AutoCorrelation(c=c)
    self_attention_layer = AutoCorrelationLayer(
        autocorrelation=self_autocorrelation,
        d_model=d_model,
        n_heads=n_heads
    )

    cross_autocorrelation = AutoCorrelation(c=c)
    cross_attention_layer = AutoCorrelationLayer(
        autocorrelation=cross_autocorrelation,
        d_model=d_model,
        n_heads=n_heads
    )

    decoder_layer = DecoderLayer(
        self_attention_layer=self_attention_layer,
        cross_attention_layer=cross_attention_layer,
        d_model=d_model,
        c_out=c_out,
        d_ff=d_ff,
        moving_avg=moving_avg,
        dropout=dropout
    )

    decoder_layers.append(decoder_layer)

decoder = Decoder(
    decoder_layers=decoder_layers,
    norm_layer=nn.LayerNorm(d_model),
    projection=None
).to(device)

seasonal_out, trend_out = decoder(x, cross, trend)

print("decoder input seasonal shape:", x.shape)
print("encoder output shape:        ", cross.shape)
print("initial trend shape:         ", trend.shape)
print("seasonal output shape:       ", seasonal_out.shape)
print("trend output shape:          ", trend_out.shape)

Device: mps
decoder input seasonal shape: torch.Size([4, 144, 512])
encoder output shape:         torch.Size([4, 96, 512])
initial trend shape:          torch.Size([4, 144, 7])
seasonal output shape:        torch.Size([4, 144, 512])
trend output shape:           torch.Size([4, 144, 7])


In [10]:
#### TESTING autoformer.py
import sys
import os

sys.path.append(os.path.abspath(".."))

import torch
from models.autoformer import Autoformer



# Select device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)

config = {
    "seq_len": 96,
    "label_len": 48,
    "pred_len": 96,

    "enc_in": 7,
    "dec_in": 7,
    "c_out": 7,

    "d_model": 512,
    "n_heads": 8,
    "d_ff": 2048,
    "enc_layers": 2,
    "dec_layers": 1,
    "moving_avg": 25,
    "c": 1,
    "dropout": 0.05,
    "freq": "t",
}

B = 4
seq_len = config["seq_len"]
label_len = config["label_len"]
pred_len = config["pred_len"]
dec_len = label_len + pred_len

enc_in = config["enc_in"]
dec_in = config["dec_in"]

# For freq="t", our TimeFeatureEmbedding expects 5 time features
time_features = 5

x_enc = torch.randn(B, seq_len, enc_in).to(device)
x_mark_enc = torch.randn(B, seq_len, time_features).to(device)

# x_dec is currently kept only for API consistency in our Autoformer forward
x_dec = torch.randn(B, dec_len, dec_in).to(device)
x_mark_dec = torch.randn(B, dec_len, time_features).to(device)

model = Autoformer(config).to(device)

out = model(
    x_enc=x_enc,
    x_mark_enc=x_mark_enc,
    x_dec=x_dec,
    x_mark_dec=x_mark_dec
)

print("x_enc shape:      ", x_enc.shape)
print("x_mark_enc shape: ", x_mark_enc.shape)
print("x_dec shape:      ", x_dec.shape)
print("x_mark_dec shape: ", x_mark_dec.shape)
print("output shape:     ", out.shape)


Device: mps
x_enc shape:       torch.Size([4, 96, 7])
x_mark_enc shape:  torch.Size([4, 96, 5])
x_dec shape:       torch.Size([4, 144, 7])
x_mark_dec shape:  torch.Size([4, 144, 5])
output shape:      torch.Size([4, 96, 7])



## Rolling windows e batch nel DataLoader

Quando lavoriamo con una serie temporale lunga, Autoformer non prende tutta la serie in una volta sola. La serie viene divisa in tante **finestre scorrevoli**, cioè tanti esempi del tipo:  
passato osservato → futuro da prevedere

Nel nostro caso usiamo:

```text
seq_len = 96      # passato dato all'encoder
label_len = 48    # parte nota data al decoder
pred_len = 96     # futuro da prevedere
```
Indichiamo con:

- $I = \text{seq\_len}$ = lunghezza del passato dato all'encoder
- $L = \text{label\_len}$ = parte iniziale del decoder che contiene passato noto
- $O = \text{pred\_len}$ = orizzonte futuro da prevedere

Per ogni indice iniziale $$i$$ costruiamo una finestra composta da:

### Encoder input
$$
x_{\text{enc}} = x[i : i+I]
$$

### Decoder input/target lungo
$$
x_{\text{dec}} = x[i+I-L : i+I+O]
$$

Questa sequenza contiene:

- gli ultimi $$L$$ punti noti del passato
- i successivi $$O$$ punti futuri


### Schema rolling window

Immaginiamo la serie temporale completa come una lunga linea:

```text
serie temporale completa
|----------------------------------------------------------------------------------------------|
0                                                                                              T


finestra 1
|---------------- seq_x ----------------|---------------- target ----------------|
0                                      96                                      192


finestra 2
 |---------------- seq_x ----------------|---------------- target ----------------|
 1                                      97                                      193


finestra 3
  |---------------- seq_x ----------------|---------------- target ----------------|
  2                                      98                                      194


...


ultima finestra valida
                                                        |---------------- seq_x ----------------|---------------- target ----------------|
                                                        i                                      i+96                                  i+192
                                                                                                                              ≤ T
```

Quindi ogni finestra è ottenuta spostandosi in avanti lungo la serie, col vincolo {i + seq_len + pred_len <= lunghezza della serie}.

Questa è una logica di tipo **rolling window**, perché la finestra mantiene lunghezza fissa e scorre nel tempo.


### Cosa contiene una singola finestra

Per ogni indice `i=[0,...,len(self.data_x) - seq_len - pred_len]`, il Dataset costruisce una finestra sulla serie temporale. Con:

```text
seq_len = 96
label_len = 48
pred_len = 96
```

abbiamo:

```text
timeline della finestra

i                         i+48                    i+96                         i+192
|--------------------------|------------------------|-----------------------------|
                           |<----- label_len ------>|<-------- pred_len --------->|
|<----------- seq_x = 96 punti passati ------------>|                             
                           |<------------- seq_y = 144 punti -------------------->|
                                                    |<------ target = 96 -------->|
```

Quindi:

```text
seq_x = data[i : i+96]
```

è il passato dato all’encoder.

```text
seq_y = data[i+48 : i+192]
```

contiene:

```text
data[i+48 : i+96]    → ultimi 48 punti noti
data[i+96 : i+192]   → 96 punti futuri veri
```

Il target usato nella loss è solo la parte finale di `seq_y`:

```text
target = data[i+96 : i+192]
```

Nel codice, su un batch, lo otteniamo con:

```python
target = batch_y[:, -pred_len:, :]
```

Quindi `seq_y` è più lungo del target perché contiene sia la parte nota per il decoder sia il futuro vero da confrontare con la previsione.

----

### e.g. schema delle finestre: cosa entra in encoder/decoder e cosa esce

Nel nostro caso:

- `seq_len = 96`
- `label_len = 48`
- `pred_len = 96`

Quindi, in notazione del paper:

- `I = seq_len = 96`
- `I/2 = label_len = 48`
- `O = pred_len = 96`


#### 1) `i = 0`

```text
timeline globale della finestra
0                48                96                               192
|----------------|-----------------|---------------------------------|

encoder input:
x_enc = seq_x
[----------------------------------)
0                                  96                            

decoder input:
x_dec = seq_y
                 [---- 48 noti ----][----------- 96 futuri -----------)
                 48                                                  192

decoder output:
y_hat
                                   [--------- 96 predetti ------------)
                                   96                                192

target per la loss:
seq_y[-pred_len:]
                                   [------------ 96 veri -------------)
                                   96                                192
```                                   

#### 2) lunghezza di ETTm2

Con lo split ufficiale degli autori, per ETTm2 usiamo:

```python
num_train = 12 * 30 * 24 * 4
```

quindi:

```text
num_train = 34560
```

Nel caso `flag="train"`:

```python
border1 = 0
border2 = num_train
```

quindi:

```python
self.data_x = data[0:34560]
```

Perciò:

```text
len(self.data_x) = 34560
```

Nel metodo `__getitem__`, `index` indica il punto da cui parte una finestra.

L’ultimo `index` valido è:

```text
len(self.data_x) - seq_len - pred_len
```

Nel nostro caso:

```text
34560 - 96 - 96 = 34368
```

Quindi:

```text
index va da 0 a 34368
```

Il numero totale di finestre è:

```text
34369
```

perché contiamo anche `index = 0`. Per l’ultima finestra abbiamo:

```text
index = 34368
```

quindi:

```python
seq_x  = data[34368 : 34464]
seq_y  = data[34416 : 34560]
target = data[34464 : 34560]
```

Questa finestra è valida perché finisce esattamente a `34560`, cioè alla fine del blocco di training, senza uscire fuori dalla serie.


### Cosa succede in un batch

Una singola finestra è un singolo esempio del Dataset.

Il `DataLoader` prende più finestre e le mette insieme in un batch.

Se:

```text
batch_size = 32
```

allora un batch contiene 32 finestre diverse:

```text
batch

finestra 1  → previsione futuro 1
finestra 2  → previsione futuro 2
finestra 3  → previsione futuro 3
...
finestra 32 → previsione futuro 32
```

Quindi il modello non fa una sola previsione, ma fa **32 previsioni in parallelo**.


### Shape del batch

Per una singola finestra avevamo:

```text
seq_x:      [96, 7]
seq_y:      [144, 7]
seq_x_mark: [96, 5]
seq_y_mark: [144, 5]
```

Quando il DataLoader costruisce un batch da 32 finestre, aggiunge una dimensione davanti:

```text
batch_x:      [32, 96, 7]
batch_y:      [32, 144, 7]
batch_x_mark: [32, 96, 5]
batch_y_mark: [32, 144, 5]
```

Dove:

```text
32  → numero di finestre nel batch
96  → lunghezza del passato encoder
144 → label_len + pred_len
7   → numero di variabili della serie
5   → numero di feature temporali
```

### Output del modello

Autoformer prende il batch:

```python
output = model(batch_x, batch_x_mark, batch_y, batch_y_mark)
```

e produce una previsione per ciascuna delle 32 finestre:

```text
output: [32, 96, 7]
```

Questo significa:

```text
32 finestre
× 96 istanti futuri previsti
× 7 variabili
```

Il target vero ha la stessa shape:

```python
target = batch_y[:, -pred_len:, :]
```

```text
target: [32, 96, 7]
```

Quindi la loss confronta:

```text
output: [32, 96, 7]
target: [32, 96, 7]
```


## Disegno intuitivo

Se ad esempio:

- $$I = 96$$
- $$L = 48$$
- $$O = 96$$

allora una finestra è fatta così:

```text
serie lunga:
|--------------------------------------------------------------- ... ------|

encoder input:
|-------------------- 96 passi --------------------|

decoder input / seq_y:
                        |------ 48 noti ------|-------- 96 futuri --------|

target finale:
                                              |-------- 96 futuri --------|

In [11]:
### TESTING dataset
import pandas as pd

df = pd.read_csv("../data/raw/ETTm2.csv")

print(df.shape)
print(df.head())
print(df.columns)

(69680, 8)
                  date       HUFL    HULL       MUFL   MULL   LUFL   LULL  \
0  2016-07-01 00:00:00  41.130001  12.481  36.535999  9.355  4.424  1.311   
1  2016-07-01 00:15:00  39.622002  11.309  35.543999  8.551  3.209  1.258   
2  2016-07-01 00:30:00  38.868000  10.555  34.365002  7.586  4.435  1.258   
3  2016-07-01 00:45:00  35.518002   9.214  32.569000  8.712  4.435  1.215   
4  2016-07-01 01:00:00  37.528000  10.136  33.936001  7.532  4.435  1.215   

          OT  
0  38.661999  
1  38.223000  
2  37.344002  
3  37.124001  
4  37.124001  
Index(['date', 'HUFL', 'HULL', 'MUFL', 'MULL', 'LUFL', 'LULL', 'OT'], dtype='str')


In [12]:
import sys
sys.path.append("..")

from data_provider.data_loader import ETTDataset

train_dataset = ETTDataset(
    data_path="../data/raw/ETTm2.csv",
    flag="train",
    seq_len=96,
    label_len=48,
    pred_len=96,
    features="M",
    target="OT",
    scale=True
)

print("Number of training windows:", len(train_dataset))



Number of training windows: 34369


In [13]:
sample = train_dataset[0]

seq_x, seq_y, seq_x_mark, seq_y_mark = sample

print("seq_x shape:", seq_x.shape)
print("seq_y shape:", seq_y.shape)
print("seq_x_mark shape:", seq_x_mark.shape)
print("seq_y_mark shape:", seq_y_mark.shape)

seq_x shape: torch.Size([96, 7])
seq_y shape: torch.Size([144, 7])
seq_x_mark shape: torch.Size([96, 5])
seq_y_mark shape: torch.Size([144, 5])


In [14]:
### TESTING data_loader.py
from data_provider.data_loader import get_data_loader

train_dataset, train_loader = get_data_loader(
    data_path="../data/raw/ETTm2.csv",
    flag="train",
    seq_len=96,
    label_len=48,
    pred_len=96,
    features="M",
    target="OT",
    batch_size=32,
    shuffle=True,
    scale=True
)

batch = next(iter(train_loader)) ### VUOL DIRE PRENDIMI IL PRIMO BATCH PRODOTTO DAL DATALOADER

batch_x, batch_y, batch_x_mark, batch_y_mark = batch
target = batch_y[:, -config["pred_len"]:, :]

print("batch_x shape:", batch_x.shape)
print("batch_x_mark shape:", batch_x_mark.shape)
print("batch_y_mark shape:", batch_y_mark.shape)
print("batch_y shape:", batch_y.shape)
print("target shape:", target.shape)

batch_x shape: torch.Size([32, 96, 7])
batch_x_mark shape: torch.Size([32, 96, 5])
batch_y_mark shape: torch.Size([32, 144, 5])
batch_y shape: torch.Size([32, 144, 7])
target shape: torch.Size([32, 96, 7])


## Primo test reale: forward + loss + primo update

In questa cella vogliamo verificare che il modello Autoformer funzioni non più su tensori casuali, ma su un **batch reale** costruito dal dataset ETTm2.

Il flusso che stiamo testando è:

```text
ETTm2.csv
→ ETTDataset
→ DataLoader
→ batch reale
→ Autoformer
→ output
→ loss
→ backward
→ update dei pesi
```

---

### 1. Batch reale dal DataLoader

Dal `DataLoader` prendiamo un batch:

```python
batch = next(iter(train_loader))
```

Il batch contiene 32 finestre temporali:

```text
batch_size = 32
```

Ogni batch viene diviso in:

```python
batch_x, batch_y, batch_x_mark, batch_y_mark = batch
```

dove:

```text
batch_x      → input dell'encoder
batch_y      → input del decoder + target futuro
batch_x_mark → time features dell'encoder
batch_y_mark → time features del decoder
```

Le shape attese sono:

```text
batch_x:      [32, 96, 7]
batch_y:      [32, 144, 7]
batch_x_mark: [32, 96, 5]
batch_y_mark: [32, 144, 5]
```

---

### 2. Target vero

Il modello deve prevedere solo gli ultimi `pred_len` punti futuri.

Dato che:

```text
pred_len = 96
```

il target vero viene estratto da `batch_y` prendendo solo la parte finale:

```python
target = batch_y[:, -pred_len:, :]
```

Quindi:

```text
target: [32, 96, 7]
```

In formule:

$$
\text{target} = batch\_y[:, -O:, :]
$$

dove:

$$
O = pred\_len = 96
$$

---

### 3. Forward del modello

Autoformer riceve:

```python
output = model(batch_x, batch_x_mark, batch_y, batch_y_mark)
```

In formula:

$$
\hat{Y} = f_{\theta}(X_{enc}, X_{mark,enc}, X_{dec}, X_{mark,dec})
$$

dove:

- $f_{\theta}$ è Autoformer;
- $\theta$ sono i pesi del modello;
- $\hat{Y}$ è la previsione prodotta dal modello.

L’output atteso è:

```text
output: [32, 96, 7]
```

cioè:

```text
32 finestre
× 96 istanti futuri
× 7 variabili
```

---

### 4. Loss MSE

Confrontiamo la previsione `output` con il target vero:

```python
loss = criterion(output, target)
```

La loss usata è la MSE:

$$
\mathcal{L}(\theta)
=
\frac{1}{M}
\sum_{m=1}^{M}
(\hat{y}_m - y_m)^2
$$

dove $$M$$ è il numero totale di valori confrontati nel batch.

Nel nostro caso la loss viene calcolata su:

```text
32 × 96 × 7
```

valori.

---

### 5. Primo update dei pesi

Dopo aver calcolato la loss, testiamo anche un singolo passo di training:

```python
optimizer.zero_grad()
output = model(batch_x, batch_x_mark, batch_y, batch_y_mark)
loss = criterion(output, target)
loss.backward()
optimizer.step()
```

Questo fa:

```text
azzeramento gradienti
→ forward
→ calcolo loss
→ backpropagation
→ aggiornamento pesi
```

In formula semplificata, l’optimizer aggiorna i pesi così:

$$
\theta_{new}
=
\theta_{old}
-
\eta \nabla_{\theta}\mathcal{L}(\theta)
$$

dove:

- $\eta$ è il learning rate;
- $\nabla_{\theta}\mathcal{L}(\theta)$ è il gradiente della loss rispetto ai pesi.

---

### Obiettivo del test

Questo non è ancora il training completo.

È solo una prova per controllare che:

```text
batch reale → modello → output → loss → backward → update
```

funzioni senza errori.

Se questo test passa, significa che il modello è pronto per essere inserito dentro una vera training loop.

In [15]:
### TESTING della prima forward reale (non su tensori random) di un batch di dati reali
import sys
import torch

sys.path.append("..")

from data_provider.data_loader import get_data_loader
from models.autoformer import Autoformer



# seleziono il device
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Device:", device)



# ricreo il dataloader
train_dataset, train_loader = get_data_loader(
    data_path="../data/raw/ETTm2.csv",
    flag="train",
    seq_len=96,
    label_len=48,
    pred_len=96,
    features="M",
    target="OT",
    batch_size=32,
    shuffle=True,
    scale=True
)


batch = next(iter(train_loader))
batch_x, batch_y, batch_x_mark, batch_y_mark = batch


# sposto tutto sul device corretto
batch_x = batch_x.to(device)
batch_x_mark = batch_x_mark.to(device)
batch_y = batch_y.to(device)
batch_y_mark = batch_y_mark.to(device)
target = target.to(device)



# ricreo il modello
config = {
    "seq_len": 96,
    "label_len": 48,
    "pred_len": 96,

    "enc_in": 7,
    "dec_in": 7,
    "c_out": 7,

    "d_model": 512,
    "n_heads": 8,
    "enc_layers": 2,
    "dec_layers": 1,
    "d_ff": 2048,

    "moving_avg": 25,
    "c": 1,
    "dropout": 0.05,
    "freq": "t",
}

target = batch_y[:, -config["pred_len"]:, :]
model = Autoformer(config).to(device)



# faccio la forward pass
output = model(batch_x, batch_x_mark, batch_y, batch_y_mark)

print("output shape:", output.shape)
print("target shape:", target.shape)



# calcolo la loss
criterion = torch.nn.MSELoss()

loss = criterion(output, target)

print("loss:", loss.item())



# calcolo gradients e faccio l'ottimizzazione
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.0001
)
model.train()
optimizer.zero_grad()
output = model(batch_x, batch_x_mark, batch_y, batch_y_mark)
loss = criterion(output, target)
loss.backward()
optimizer.step()

print("Training step completed")
print("loss:", loss.item())

Device: mps
output shape: torch.Size([32, 96, 7])
target shape: torch.Size([32, 96, 7])
loss: 9.192468643188477
Training step completed
loss: 9.295429229736328


In [16]:
### TESTING mini training loop con pochi batch di dati reali
num_debug_batches = 50

model.train()

for batch_idx, batch in enumerate(train_loader):
    batch_x, batch_y, batch_x_mark, batch_y_mark = batch

    batch_x = batch_x.to(device)
    batch_x_mark = batch_x_mark.to(device)
    batch_y = batch_y.to(device)
    batch_y_mark = batch_y_mark.to(device)

    target = batch_y[:, -config["pred_len"]:, :]

    optimizer.zero_grad()

    output = model(batch_x, batch_x_mark, batch_y, batch_y_mark)

    loss = criterion(output, target)

    loss.backward()

    optimizer.step()

    print(f"Batch {batch_idx + 1}/{num_debug_batches} - loss: {loss.item():.4f}")

    if batch_idx + 1 == num_debug_batches:
        break

Batch 1/50 - loss: 6.7388
Batch 2/50 - loss: 5.2761
Batch 3/50 - loss: 3.3486
Batch 4/50 - loss: 2.7265
Batch 5/50 - loss: 1.8499
Batch 6/50 - loss: 1.4935
Batch 7/50 - loss: 1.5015
Batch 8/50 - loss: 1.3451
Batch 9/50 - loss: 1.6385
Batch 10/50 - loss: 1.8823
Batch 11/50 - loss: 1.9553
Batch 12/50 - loss: 1.8962
Batch 13/50 - loss: 1.4262
Batch 14/50 - loss: 1.4968
Batch 15/50 - loss: 1.3204
Batch 16/50 - loss: 1.3404
Batch 17/50 - loss: 1.0423
Batch 18/50 - loss: 1.1017
Batch 19/50 - loss: 0.9223
Batch 20/50 - loss: 1.0123
Batch 21/50 - loss: 0.9026
Batch 22/50 - loss: 0.9241
Batch 23/50 - loss: 0.8583
Batch 24/50 - loss: 0.8498
Batch 25/50 - loss: 1.0292
Batch 26/50 - loss: 0.9968
Batch 27/50 - loss: 1.0439
Batch 28/50 - loss: 0.7184
Batch 29/50 - loss: 0.8656
Batch 30/50 - loss: 0.8171
Batch 31/50 - loss: 0.8409
Batch 32/50 - loss: 0.6430
Batch 33/50 - loss: 0.9336
Batch 34/50 - loss: 0.7408
Batch 35/50 - loss: 0.6009
Batch 36/50 - loss: 0.6683
Batch 37/50 - loss: 0.8465
Batch 38/5

In [17]:
### trasformiamo il mini training loop in una funzione di training vera e propria
# il test deve essere fatto solo una volta. se lo vuoi far rigirare con un numero di batch diverso devi ricreare il modello
def train_one_epoch(model, train_loader, criterion, optimizer, device, pred_len, max_batches=None):
    """
    Train the model for one epoch.

    If max_batches is not None, the training stops after that number of batches.
    This is useful for debugging.
    """

    model.train()

    total_loss = 0.0
    num_batches = 0

    for batch_idx, batch in enumerate(train_loader):
        batch_x, batch_y, batch_x_mark, batch_y_mark = batch

        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)
        batch_x_mark = batch_x_mark.to(device)
        batch_y_mark = batch_y_mark.to(device)

        target = batch_y[:, -pred_len:, :]

        optimizer.zero_grad()

        output = model(
            batch_x,
            batch_x_mark,
            batch_y,
            batch_y_mark
        )

        loss = criterion(output, target)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        # Prima stampavamo la loss batch per batch.
        # Ora accumuliamo le loss e alla fine facciamo la media.
        total_loss += loss.item()
        num_batches += 1

        if max_batches is not None and num_batches >= max_batches:
            break

    average_loss = total_loss / num_batches

    return average_loss



# ora possiamo usare la funzione di training per fare un mini training loop di debug su 20 batch
train_loss = train_one_epoch(
    model=model,
    train_loader=train_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=device,
    pred_len=config["pred_len"],
    max_batches=50
)

print("Train loss:", train_loss) # stampa la loss media sui 20 batch di debug

Train loss: 0.913605352640152


## Fase di valutazione: validation

Dopo aver definito la funzione di training, costruiamo anche una funzione di **validation**.

La validation serve a misurare quanto il modello funziona su dati che non sta usando per aggiornare i pesi.

Durante il training facciamo:

```text
forward → loss → backward → update pesi
```

Durante la validation invece facciamo solo:

```text
forward → loss
```

Quindi nella validation:

- il modello produce una previsione;
- confrontiamo la previsione con il target vero;
- calcoliamo la loss;
- **non facciamo backpropagation**;
- **non aggiorniamo i pesi**.

Per questo usiamo:

```python
model.eval()
```

che mette il modello in modalità valutazione, e:

```python
with torch.no_grad():
```

che dice a PyTorch di non calcolare i gradienti.

Il batch ha sempre la stessa struttura:

```python
batch_x, batch_y, batch_x_mark, batch_y_mark = batch
```

Il target vero viene estratto dalla parte finale di `batch_y`:

```python
target = batch_y[:, -pred_len:, :]
```

Il modello produce:

```python
output = model(batch_x, batch_x_mark, batch_y, batch_y_mark)
```

e la validation loss viene calcolata come:

```python
loss = criterion(output, target)
```

In formula:

$$
\hat{Y} = f_{\theta}(X)
$$

$$
\mathcal{L}_{val}
=
MSE(\hat{Y}, Y)
$$

La differenza fondamentale è che in validation i parametri $$\theta$$ restano fissi: **sono quelli calcolati nell'ultimo training dell'ultimo batch (calcolato sul 50esimo batch con train_loss = train_one_epoch() nella cella precedente), infatti se rilanci piu volte questa cella non succede nulla perche usa sempre quei pesi e non ci sono "sovrascrizioni"**

Quindi:

```text
training   → misura errore e aggiorna i pesi
validation → misura errore senza aggiornare i pesi
```

La validation loss ci serve per capire se il modello sta imparando in modo utile oppure se sta solo adattandosi troppo al training set.

In [18]:
def validate(model, val_loader, criterion, device, pred_len, max_batches=None):
    """
    Evaluate the model on the validation set.

    If max_batches is not None, the validation stops after that number of batches.
    This is useful for debugging.
    """

    model.eval() ### mette il modello in modalità valutazione. Per esempio, il Dropout non viene più applicato come durante il training

    total_loss = 0.0
    num_batches = 0

    with torch.no_grad(): ### dice a PyTorch: “non salvare il grafo computazionale e non calcolare gradienti”. 
                            ### Questo rende la validation più leggera e veloce, perché non dobbiamo aggiornare i pesi
        for batch_idx, batch in enumerate(val_loader):
            batch_x, batch_y, batch_x_mark, batch_y_mark = batch

            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            batch_x_mark = batch_x_mark.to(device)
            batch_y_mark = batch_y_mark.to(device)

            target = batch_y[:, -pred_len:, :]

            output = model(
                batch_x,
                batch_x_mark,
                batch_y,
                batch_y_mark
            )

            loss = criterion(output, target)

            total_loss += loss.item()
            num_batches += 1

            if max_batches is not None and num_batches >= max_batches:
                break

    average_loss = total_loss / num_batches

    return average_loss



val_dataset, val_loader = get_data_loader(
    data_path="../data/raw/ETTm2.csv",
    flag="val",
    seq_len=96,
    label_len=48,
    pred_len=96,
    features="M",
    target="OT",
    batch_size=32,
    shuffle=False,
    scale=True
)

print("Number of validation windows:", len(val_dataset))




val_loss = validate(
    model=model,
    val_loader=val_loader,
    criterion=criterion,
    device=device,
    pred_len=config["pred_len"],
    max_batches=20
)

print("Validation loss:", val_loss)

Number of validation windows: 11425
Validation loss: 1.100358384847641


In [19]:
### test piu grosso per vedere due epoche quasi grandi come nella realtà, ma con un numero di batch limitato per non farlo durare troppo
model = Autoformer(config).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.00001
)


import time

num_epochs = 3

max_train_batches = 500
max_val_batches = 100

for epoch in range(num_epochs):
    start_time = time.time()

    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        criterion=criterion,
        optimizer=optimizer,
        device=device,
        pred_len=config["pred_len"],
        max_batches=max_train_batches
    )

    val_loss = validate(
        model=model,
        val_loader=val_loader,
        criterion=criterion,
        device=device,
        pred_len=config["pred_len"],
        max_batches=max_val_batches
    )

    epoch_time = time.time() - start_time

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"train loss: {train_loss:.4f} | "
        f"val loss: {val_loss:.4f} | "
        f"time: {epoch_time:.2f}s"
    )

Epoch 1/3 | train loss: 1.1360 | val loss: 0.2940 | time: 174.90s
Epoch 2/3 | train loss: 38.3498 | val loss: 10.6557 | time: 176.55s
Epoch 3/3 | train loss: 241.3831 | val loss: 21.4852 | time: 174.55s
